# Phase 1 Slice D: Snapshot Render API (PNG)

This notebook validates the hardened snapshot contract for `POST /render/image`:

- Stateful render via `view_id`
- Stateless render via inline `view_state`
- PNG delivery modes: `inline_base64` and `file_path`
- Safe output-path behavior under repository `output/`
- One-of validation for `view_id` vs `view_state`


In [1]:
from __future__ import annotations

import base64
import tempfile
import uuid
from io import BytesIO
from pathlib import Path

import numpy as np
import zarr
from fastapi.testclient import TestClient
from PIL import Image

from lucida.client import LucidaClient
from lucida.server.app import create_app
from lucida.service.dataset_service import DatasetService

from lucida.service import dataset_service as dataset_service_module

output_root = Path(dataset_service_module.__file__).resolve().parents[3] / 'output'
output_root.mkdir(parents=True, exist_ok=True)


In [2]:
def create_render_omezarr(uri: str) -> str:
    root = zarr.open_group(store=uri, mode='w')

    shape_level0 = (1, 3, 4, 5, 6)
    data_level0 = np.zeros(shape_level0, dtype=np.uint16)
    for c in range(shape_level0[1]):
        for z in range(shape_level0[2]):
            for y in range(shape_level0[3]):
                for x in range(shape_level0[4]):
                    data_level0[0, c, z, y, x] = np.uint16((c * 1000) + (z * 100) + (y * 10) + x)

    data_level1 = data_level0[:, :, ::2, ::2, ::2]

    root.create_array('0', data=data_level0, chunks=(1, 1, 2, 3, 3), overwrite=True)
    root.create_array('1', data=data_level1, chunks=(1, 1, 1, 2, 2), overwrite=True)

    root.attrs['multiscales'] = [
        {
            'name': 'primary',
            'axes': [
                {'name': 't', 'type': 't'},
                {'name': 'c', 'type': 'c'},
                {'name': 'z', 'type': 'z'},
                {'name': 'y', 'type': 'y'},
                {'name': 'x', 'type': 'x'},
            ],
            'datasets': [
                {'path': '0', 'coordinateTransformations': [{'type': 'scale', 'scale': [1, 1, 1, 1, 1]}]},
                {'path': '1', 'coordinateTransformations': [{'type': 'scale', 'scale': [1, 1, 2, 2, 2]}]},
            ],
        }
    ]
    root.attrs['omero'] = {
        'channels': [
            {'index': 0, 'label': 'c0', 'color': 'ffffff', 'window': {'start': 0, 'end': 500}},
            {'index': 1, 'label': 'c1', 'color': 'ff0000', 'window': {'start': 0, 'end': 1500}},
            {'index': 2, 'label': 'c2', 'color': '00ff00', 'window': {'start': 0, 'end': 2500}},
        ]
    }
    return uri


In [3]:
tmp_dir = Path(tempfile.mkdtemp(prefix='lucida-snapshot-'))
dataset_uri = create_render_omezarr(str(tmp_dir / 'render.zarr'))

service = DatasetService()
app = create_app(dataset_service=service)
http_client = TestClient(app)
client = LucidaClient(client=http_client)

session = client.create_session()
opened = client.open_dataset(uri=dataset_uri, session_id=session.session_id)
created = client.create_view(
    dataset_id=opened.dataset_summary.dataset_id,
    session_id=session.session_id,
    mode='2d',
)
view_id = created.view_state.view_id

assert opened.schema_version == 1
assert bool(opened.dataset_summary.dataset_id)
assert created.view_state.mode == '2d'
view_id


'view_c05a7a630c5146c9'

In [4]:
stateful_inline = client.render_image(
    view_id=view_id,
    session_id=session.session_id,
    width_px=96,
    height_px=64,
)

assert stateful_inline.schema_version == 1
assert stateful_inline.status == 'ok'
assert stateful_inline.images[0].mime == 'image/png'
assert stateful_inline.images[0].bytes_base64 is not None
decoded = Image.open(BytesIO(base64.b64decode(stateful_inline.images[0].bytes_base64))).convert('RGBA')
assert decoded.size == (96, 64)
stateful_inline


RenderImageResponse(schema_version=1, request_id='req_64dc39450d154e27', render_id='ren_62806e282e034a4a', status='ok', completion=1.0, view_id='view_c05a7a630c5146c9', state_hash='00581c8447369e026f298e246423ea4c29a44e768b897ba6ea1f516cf6e2f80e', state_version=0, images=[RenderImageArtifact(role='main', mime='image/png', width_px=96, height_px=64, delivery='inline_base64', bytes_base64='iVBORw0KGgoAAAANSUhEUgAAAGAAAABACAYAAADlNHIOAAAA60lEQVR4nO3bQUrFMBRA0Rex/ev5a3GXrsUNOVH5FcEdXCPnUAgZNeSS0EnXzHwOmafu1QjwBzgBMQFiAsQEiAkQEyAmQEyAmAAxAWICxASICRATICZATICYADEBYgLEBIgJEBMgJkBMgJgAMQFiz7Oh17eZtdas9Zh9jy/3j9nRlgFu53EFmOv5CfE+O9ozwO24Nl6AKsB5XuPva2hXe56A8/g3AR4r94dMyGdoTICYADEBYgLEBIgJEBMgJkBMgJgAMQFiAsQEiAkQEyAmQEyAmAAxAWICxASICRATICZATICYADEBYgJM6ws6EgfeiV/RcgAAAABJRU5ErkJggg==', file_path=None, sha256='2a7177d62bb99ad4d57e31b5ff633283fc053e3593ae309c5693444aa937f3bd')], meta=RenderMeta(dataset_id='ds_e6fa656064fbb9c2', multiscale_name='primary', pyramid_level_used=0, selectors_applied=[Axi

In [5]:
explicit_relative = f"snapshots/notebook-explicit-{uuid.uuid4().hex}.png"
stateful_file = client.render_image(
    view_id=view_id,
    session_id=session.session_id,
    width_px=80,
    height_px=56,
    delivery='file_path',
    file_path=explicit_relative,
)
explicit_path = Path(stateful_file.images[0].file_path or '')

auto_file = client.render_image(
    view_id=view_id,
    session_id=session.session_id,
    width_px=72,
    height_px=48,
    delivery='file_path',
)
auto_path = Path(auto_file.images[0].file_path or '')

assert explicit_path.exists()
assert explicit_path.is_relative_to(output_root)
assert auto_path.exists()
assert auto_path.is_relative_to(output_root / 'snapshots')
(stateful_file.images[0].file_path, auto_file.images[0].file_path)


('/Users/austin/GitHub/lucida/output/snapshots/notebook-explicit-1173ca5245274cbd837e76abb6ceec58.png',
 '/Users/austin/GitHub/lucida/output/snapshots/ren_efa84c6985ca43d3.png')

In [6]:
view_state = client.get_view(view_id=view_id, session_id=session.session_id).view_state
stateless_inline = client.render_image(
    view_state=view_state,
    session_id=session.session_id,
    width_px=88,
    height_px=60,
)

assert stateless_inline.schema_version == 1
assert stateless_inline.status == 'ok'
assert stateless_inline.view_id is None
assert stateless_inline.state_version is None
assert stateless_inline.images[0].mime == 'image/png'
decoded_stateless = Image.open(BytesIO(base64.b64decode(stateless_inline.images[0].bytes_base64))).convert('RGBA')
assert decoded_stateless.size == (88, 60)
stateless_inline


RenderImageResponse(schema_version=1, request_id='req_3c150e143a544e07', render_id='ren_923095e5a9534c2b', status='ok', completion=1.0, view_id=None, state_hash='00581c8447369e026f298e246423ea4c29a44e768b897ba6ea1f516cf6e2f80e', state_version=None, images=[RenderImageArtifact(role='main', mime='image/png', width_px=88, height_px=60, delivery='inline_base64', bytes_base64='iVBORw0KGgoAAAANSUhEUgAAAFgAAAA8CAYAAADi8H14AAAA4klEQVR4nO3a0UnEQBRA0TdisvVYy3ZpLTbkj8pGFmzgCgPnMDDkZwiXx5CPrJn5HjIv3dEI/A9McEzgmMAxgWMCxwSOCRwTOCZwTOCYwDGBYwLHBI4JHBM4JnBM4JjAMYFjAscEjgkcEzj2Oht4/5hZa81aj6ff/f72NTvYIvDtPK7Ac61n6M/ZwR6Bb8cVVuAq8Hle+99rYhd7TPB5bBv48ab+Dw75TIsJHBM4JnBM4JjAMYFjAscEjgkcEzgmcEzgmMAxgWMCxwSOCRwTOCZwTOCYwDGBYwLHBI4JHBN4Wj8IEgfWLaj+BgAAAABJRU5ErkJggg==', file_path=None, sha256='cc53c58f4ad1f3dfc0a3d179f391d5b2bd84b2d6ca7e17acfcc5e576160eb29e')], meta=RenderMeta(dataset_id='ds_e6fa656064fbb9c2', multiscale_name='primary', pyramid_level_used=0, selectors_applied=[AxisSelector(axis='t', kind='in

In [7]:
raw_stateless = http_client.post(
    '/render/image',
    json={
        'schema_version': 1,
        'view_state': view_state.model_dump(mode='json'),
        'output': {
            'format': 'png',
            'delivery': 'inline_base64',
            'width_px': 64,
            'height_px': 40,
        },
    },
)
assert raw_stateless.status_code == 200
raw_payload = raw_stateless.json()
assert raw_payload['schema_version'] == 1
assert raw_payload['status'] == 'ok'
assert 'view_id' not in raw_payload
assert 'state_version' not in raw_payload
raw_payload['images'][0]['mime']


'image/png'

In [8]:
invalid = http_client.post(
    '/render/image',
    json={
        'schema_version': 1,
        'view_id': view_id,
        'view_state': view_state.model_dump(mode='json'),
        'output': {
            'format': 'png',
            'delivery': 'inline_base64',
            'width_px': 32,
            'height_px': 24,
        },
    },
)
assert invalid.status_code == 422
assert invalid.json()['code'] == 'invalid_render_request'
invalid.json()


{'code': 'invalid_render_request',
 'message': 'Render request must provide exactly one of view_id or view_state.',
 'details': {'errors': [{'type': 'value_error',
    'loc': ['body'],
    'msg': 'Value error, Exactly one of view_id or view_state must be provided.',
    'input': {'schema_version': 1,
     'view_id': 'view_c05a7a630c5146c9',
     'view_state': {'schema_version': 1,
      'view_id': 'view_c05a7a630c5146c9',
      'session_id': 'session_6d8ae9fe7219449a',
      'created_at': '2026-02-24T00:52:18.709490Z',
      'mode': '2d',
      'datasets': [{'dataset_id': 'ds_e6fa656064fbb9c2',
        'multiscale_name': 'primary'}],
      'viewport': {'width_px': 1024, 'height_px': 1024, 'pixel_ratio': 1.0},
      'selectors': [{'axis': 't',
        'kind': 'index',
        'index': 0,
        'start': None,
        'end_exclusive': None,
        'indices': None,
        'clamp': True},
       {'axis': 'c',
        'kind': 'index',
        'index': 0,
        'start': None,
        'e

In [9]:
client.close()
http_client.close()
if explicit_path.exists():
    explicit_path.unlink()
if auto_path.exists():
    auto_path.unlink()
'cleanup complete'


'cleanup complete'

## Expected output

Verify all checks below succeed:

- `schema_version == 1` for open/render responses.
- Render responses have `status == "ok"` and `mime == "image/png"`.
- Decoded image dimensions match requested `width_px` and `height_px`.
- `delivery=file_path` writes files inside repository `output/` (explicit and auto paths).
- Stateless endpoint response omits `view_id` and `state_version`.
- Request containing both `view_id` and `view_state` fails with `invalid_render_request`.
